# HVAC Electircity Demand Analysis and Prediction

## Feature Creation

### Import Libraries



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor,RandomForestRegressor

np.random.seed(42)

In [ ]:
df_daily = pd.read_csv("./data/df_daily.csv", index_col=0)
df = pd.read_csv("./data/Load_data_01.csv")
df["Time"] = pd.to_datetime(df["Time"])
df.set_index("Time", inplace=True)
df.describe().T

In [ ]:
df_daily.describe().T


## Create daily dataframe and feature creation

In [ ]:
# Collect allavailable solar irradiation data in a day
df["solar_irridiation_positive"] = df["solar_irridiation"][df["solar_irridiation"] > 0]

In [ ]:
df["solar_irridiation_positive"].resample("D").count().value_counts() # 13 hours solar irradiation per day

In [ ]:
df["solar_irridiation"][df["solar_irridiation"] == 0].resample("D").count().value_counts()  # 11 hours no solar irradiation per day

In [ ]:
df_daily = pd.DataFrame() # create a new dataframe for daily data

df_daily["electricity_demand_values"] = ( # sum up all electricity demand values in a day
    df["electricity_demand_values"].resample("D").sum() 
)

In [ ]:
# Define the columns for mean calculation
mean_cols = [
    "air_pressure",
    "air_temperature",
    "relative_humidity",
    "wind_speed",
    "solar_irridiation_positive",
    "total_cloud_cover_percent",
]

# Calculate the mean for each of the specified columns on a daily basis
for col in mean_cols:
    df_daily[col] = df[col].resample("D").mean()

In [ ]:
daily_resampled_wind = df["wind_speed"].resample("D")
df_daily["wind_speed_range"] = (
    daily_resampled_wind.max() - daily_resampled_wind.min()
)  # calculate range of wind speed

In [ ]:
def label_wind(wind): # label wind scale
    if wind < 5:
        return "Wind scale 2"
    else:
        return "Wind scale 3"

In [ ]:
df_daily["wind_label"] = df_daily["wind_speed"].apply(label_wind) # apply label_wind function to wind speed column
df_wind = pd.get_dummies(df_daily["wind_label"])

for _ in df_wind.columns:   # add wind scale columns to df_daily
    df_daily[_] = df_wind[_].astype(int)

In [ ]:
def label_humidity(humidity):   # label humidity
    if humidity < 30:
        return "Uncomfortable Dry"  # 0
    elif humidity > 60:
        return "Uncomfortable Wet"  # 2
    else:
        return "Comfort"  # 1

In [ ]:
df_daily["relative_humidity_label"] = df_daily["relative_humidity"].apply(
    label_humidity
)
df_humidity = pd.get_dummies(df_daily["relative_humidity_label"], prefix="huimidity")
for _ in df_humidity.columns:
    df_daily[_] = df_humidity[_].astype(int)


In [ ]:
def label_temperature(temp):  # label temperature
    if temp < -10:
        return "temp_less_minus_10"  
    elif temp < 0:
        return "temp_minus_10_0"
    elif temp < 10:
        return "temp_0_10"
    elif temp < 20:
        return "temp_10_20"
    elif temp < 30:
        return "temp_20_30"
    elif temp < 40:
        return "temp_30_40"
    else:
        return "temp_greater_40"


df_daily["temp_range"] = df_daily["air_temperature"].apply(
    label_temperature
)  # apply label_temperature function to air_temperature column
df_temperature = pd.get_dummies(df_daily["temp_range"])

for _ in df_temperature.columns:
    df_daily[_] = df_temperature[_].astype(int)


In [ ]:
df_daily.drop( # drop orginal features to avoid multicollinearity
    [
        "air_temperature",
        "wind_speed",
        "relative_humidity",
        "temp_range",
        "relative_humidity_label",
        "wind_label","wind_label"
    ],
    axis=1,
    inplace=True,
)

In [ ]:
# create new features from time index
df_daily["Day_of_Week"] = df_daily.index.day_of_week
df_daily["month"] = df_daily.index.month
df_daily = pd.get_dummies(
    df_daily,
    columns=["Day_of_Week", "month"],
    prefix=["Is_Weekday","month"],
)

In [ ]:
# Convert dummy variables to integer type

for _ in [
    "Is_Weekday_0",
    "Is_Weekday_1",
    "Is_Weekday_2",
    "Is_Weekday_3",
    "Is_Weekday_4",
    "Is_Weekday_5",
    "Is_Weekday_6",
    "month_1",
    "month_2",
    "month_3",
    "month_4",
    "month_5",
    "month_6",
    "month_7",
    "month_8",
    "month_9",
    "month_10",
    "month_11",
    "month_12",
]:
    df_daily[_] = df_daily[_].astype(int)

In [ ]:
# df_daily.to_csv(
#     "./data/df_daily_feature_creation.csv"
# )  # Save the daily DataFrame with created features

## Training and Testing models

In [ ]:
# Define a function to split the dataset into training and testing sets
def train_test_set(df, start, end, split_time):
    train = df[(df.index > start) & (df.index <= split_time)]
    test = df[(df.index > split_time) & (df.index <= end)]
    X_train, y_train = (
        train.drop(["electricity_demand_values"], axis=1),
        train["electricity_demand_values"],
    )
    X_test, y_test = (
        test.drop(["electricity_demand_values"], axis=1),
        test["electricity_demand_values"],
    )
    return X_train, y_train, X_test, y_test

In [ ]:
def MAPE(y_true, y_pred):  # Calculate Mean Absolute Percentage Error
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_index = y_true != 0
    y_true = y_true[non_zero_index]
    y_pred = y_pred[non_zero_index]
    if len(y_true) == 0:
        return np.nan
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100  # in percentage

In [ ]:
def evaluate_model_performance(y_true, y_pred):  # Evaluate model performance
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = np.mean(np.abs(y_true - y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = MAPE(y_true, y_pred)
    return round(rmse, 4), round(mae, 4), round(mape, 2), round(r2, 4)


In [ ]:
def train_model(X_train, y_train, X_test, y_test, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse, mae, mape, r2 = evaluate_model_performance(y_test, y_pred)
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MAPE: {mape:.2f} %")
    print(f"R2: {r2:.4f}")
    df_performance = pd.DataFrame(
        {"RMSE": rmse, "MAE": mae, "MAPE(%)": mape, "R2": r2}, index=[0]
    )
    df_results = pd.DataFrame(
        {"Time": y_test.index, "y_test": y_test, "y_pred": y_pred}
    )
    return df_performance,df_results

In [ ]:
X_train, y_train, X_test, y_test = train_test_set(
    df_daily, "2017-01-01", "2018-11-20", "2018-06-01"
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
# Plot the train and test data
_, ax = plt.subplots(figsize=(12, 3))

ax.plot(y_train.index, y_train, label="Train")
ax.plot(y_test.index, y_test, label="Test")

plt.legend()
plt.tight_layout()

plt.show()


In [ ]:
# Define a function to plot the results
def plot_results(df_results, model_name):
    _, ax = plt.subplots(figsize=(12, 3))
    ax.plot(df_results["Time"], df_results["y_test"], label="Test")
    ax.plot(df_results["Time"], df_results["y_pred"], label="Predicted")
    ax.set_title(model_name)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Train and evaluate Decision Tree Regressor
dt_reg = DecisionTreeRegressor(max_depth=10, random_state=42)
df_performance, df_results = train_model(X_train, y_train, X_test, y_test, dt_reg)

In [ ]:
df_performance

In [ ]:
plot_results(df_results, "Decision Tree Regressor")  # Plot the results

In [ ]:
# Train and evaluate Random Forest Regressor
RF_reg = RandomForestRegressor(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)

df_performance, df_results = train_model(X_train, y_train, X_test, y_test, RF_reg)

In [ ]:
df_performance

In [ ]:
plot_results(df_results, "Random Forest Regressor")  # Plot the results

In [ ]:
# Train and evaluate AdaBoost Regressor
ada_reg = AdaBoostRegressor(
    DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
)

df_performance, df_results = train_model(X_train, y_train, X_test, y_test, ada_reg)

In [ ]:
df_performance

In [ ]:
plot_results(df_results, "AdaBoost Regressor")  # Plot the results

In [ ]:
# Plot the correlation matrix of daily data with created features
_, ax = plt.subplots(figsize=(16, 16))
sns.heatmap(df_daily.corr(), annot=True, ax=ax)
ax.set_title("Correlation Matrix of Daily Data with created features")
plt.show()

## Conclusion

- By adding more virtual features, all the models performance are improved.
- While the RMSE,MAE and R-squared are still comparably low.